### Imports & OpenEPD Setup

In [1]:
import os
import pandas as pd
import json

# import API client (requires openepd library)
from openepd.api.sync_client import OpenEpdApiClientSync

EC3_BASE_URL = "https://buildingtransparency.org/epds/"

token = os.environ['EC3_KEY'] #assumes EC3 access token is stored as environment variable

In [2]:
import urllib3
import requests
import warnings

# Disable SSL warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings('ignore', message='Unverified HTTPS request')

# Monkey-patch requests globally to disable SSL verification
original_request = requests.Session.request

def patched_request(self, method, url, **kwargs):
    kwargs['verify'] = False
    return original_request(self, method, url, **kwargs)

requests.Session.request = patched_request

print("✓ SSL verification disabled globally for all requests")

✓ SSL verification disabled globally for all requests


### Create API Client

In [3]:
client = OpenEpdApiClientSync(
        base_url="https://openepd.buildingtransparency.org/api",
        auth_token=token)

### Begin Material Query

In [20]:
# documentation here: https://docs.open-epd-forum.org/en/rest-api/#/operations/get_epds_search
omf_query = (
    '!EC3 search("ReadyMix") '
    'WHERE valid_until: > "2024-01-01" '
    'AND specs.concrete.strength_28d: >"4500 psi" '
    'AND specs.concrete.strength_28d: <"5500 psi" '
    '!pragma oMF("1.0/1")'
    )

response = client.epds.find(omf_query, page_size=100)

print(f"< Total EPDs found: {response.get_total_count()}")

< Total EPDs found: 22253


In [22]:
import json

# Get the first EPD
epd_iterator = response.iterator()
first_epd = next(epd_iterator)

# Print as formatted JSON
print(json.dumps(first_epd.dict(), indent=2, default=str))

{
  "ext": null,
  "doctype": "OpenEPD",
  "openepd_version": "0.1",
  "alt_ids": null,
  "attachments": null,
  "id": "wapbgfux",
  "product_name": "F50NAM",
  "product_sku": null,
  "product_description": "5000 FA NA",
  "product_classes": {
    "io.cqd.ec3": "Concrete >> ReadyMix",
    "EC3": "Concrete >> ReadyMix"
  },
  "product_image_small": null,
  "product_image": null,
  "version": 75160,
  "language": "en",
  "private": false,
  "declaration_url": "https://concrete.thetaepd.com/public?guid=9d9db799-0bc5-420a-bf89-792bae83e239",
  "manufacturer": null,
  "epd_developer": null,
  "epd_developer_email": null,
  "plants": [],
  "program_operator": null,
  "program_operator_doc_id": null,
  "program_operator_version": null,
  "third_party_verifier": null,
  "third_party_verification_url": null,
  "third_party_verifier_email": null,
  "date_of_issue": "2025-12-15 00:00:00+00:00",
  "valid_until": "2028-12-20 00:00:00+00:00",
  "pcr": null,
  "declared_unit": {
    "ext": null,
    

In [24]:
print("> Fetching first 5 EPDs. Might take a few seconds...")
    # fetch first 5 EPDs. The API client provides a
    # transparent iterator which would load more pages
    # as needed, but we only need first 5 objects here
epd_iterator = response.iterator()
for epd in [next(epd_iterator) for _ in range(5)]:
    # print nicely formatted result
    print("< Found EPD:")
    id_field = f"{'id             '}: {epd.id}"
    name_field = f"{'mix name       '}: {epd.product_name}"
    description_field = f"{'description    '}: {epd.product_description}"
    applicable_in = f"{'applicable in  '}: {epd.applicable_in}"
    link = f"{'See at EC3     '}: {EC3_BASE_URL}{epd.id}"

    print("\n".join([id_field, name_field, description_field, applicable_in, link]))
    print("\n")


> Fetching first 5 EPDs. Might take a few seconds...
< Found EPD:
id             : wapbgfux
mix name       : F50NAM
description    : 5000 FA NA
applicable in  : ['001']
See at EC3     : https://buildingtransparency.org/epds/wapbgfux


< Found EPD:
id             : wap55zea
mix name       : 325042442
description    : 5000 PSI 3/4LS MRWR
applicable in  : ['001']
See at EC3     : https://buildingtransparency.org/epds/wap55zea


< Found EPD:
id             : wap4eb3d
mix name       : 325045342
description    : 5000 PSI 3/4LS MRWR
applicable in  : ['001']
See at EC3     : https://buildingtransparency.org/epds/wap4eb3d


< Found EPD:
id             : wape53sg
mix name       : 4505-TEXAN
description    : 4500 FA 57
applicable in  : ['001']
See at EC3     : https://buildingtransparency.org/epds/wape53sg


< Found EPD:
id             : wap31b71
mix name       : 4505-TEXAN
description    : 4500 FA 57
applicable in  : ['001']
See at EC3     : https://buildingtransparency.org/epds/wap31b71




In [ ]:
# Initialize empty list to store filtered EPD records
filtered_epds = []

# Loop through all EPDs in the response
for epd in response.iterator():
    try:
        # Rule 1: Check if 'US' is in applicable_in array
        if not hasattr(epd, 'applicable_in') or 'US' not in epd.applicable_in:
            continue

        # Rule 2: Check if specs.concrete.cementitious is not null
        if not (hasattr(epd, 'specs') and
                hasattr(epd.specs, 'concrete') and
                hasattr(epd.specs.concrete, 'cementitious') and
                epd.specs.concrete.cementitious is not None):
            continue

        # Convert EPD to dictionary and add to list
        # TODO: Further specify which fields to save instead of saving all fields
        epd_dict = epd.dict()
        filtered_epds.append(epd_dict)

    except (AttributeError, TypeError) as e:
        # Skip records that cause errors
        continue

print(f"Total EPDs found: {response.get_total_count()}")
print(f"Filtered EPDs (US with cementitious data): {len(filtered_epds)}")

Total EPDs found: 22253
Filtered EPDs (US with cementitious data): 0


### Main EC3 Query
**This will take some time to run if page_size is not limited**

In [ ]:
# epd_param_dict = {"concrete_compressive_strength_28d":"4000 psi",
#                   "lightweight":False,
#                   "plant_geography": ["US"],
#                   "product_specific": True,
#                   "manufacturer_specific": True
#                   }

# ec3_epds.display_name_filter = ["Ready Mix"]

# ec3_epds.only_valid = False

# ec3_epds.return_fields = ["id", "date_of_issue", "cementitious", "concrete_compressive_strength_28d", "gwp", "gwp_per_category_declared_unit", "plant_geography", "lightweight", "plant_or_group"]
# ec3_epds.sort_by = "date_of_issue"
# ec3_epds.page_size = 100

# epd_records = ec3_epds.get_epds(return_all=False, params=epd_param_dict)

In [ ]:
# #filter down only the records that have the cementitious field
# epd_records_filtered = [record for record in epd_records if 'cementitious' in record]

### Save to JSON

In [ ]:
#Navigate up the directory and into the 01_raw_data folder to save the data
# os.chdir(os.path.dirname(os.getcwd()))
# os.chdir('01_raw_data')

# with open('epddata_4000psi_test.json', 'w') as f:
#     json.dump(epd_records_filtered, f, indent=4)